# Description
```
Name      : gan_2
Problem   : Classification
Type      : Deep learning
Algorithm : Convolutional neural network : GAN
Formula   : Weightes-Sum
Action    : Leaky Relu | Softmax
Loss      : sparse_categorical_crossentropy
Train     : Unsupervised
Input     : Feature and label
Output    : Classification
Dataset   : Structured : mnist
```

# General

### Import

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from keras.datasets import mnist
from keras import models, layers
from keras import optimizers
from keras.optimizers.schedules import ExponentialDecay
from keras.models import load_model

### Variable

In [ ]:
root_dir = '/Volumes/data/documents/ai_document'
output_label = ['Fake', 'Real']
colors = [(255, 0, 0), (0, 255, 0)]
noise_size = 100
n_nodes = 128 * 7 * 7
EPOCHS = 100
BATCH_SIZE = 256

# Model

### Optimizers

In [ ]:
lr_schedule = ExponentialDecay(
    initial_learning_rate=0.05,
    decay_steps=255,
    decay_rate=0.002,
    staircase=False
)

opt_sgd = optimizers.SGD(learning_rate=lr_schedule)
opt_adam = optimizers.Adam(learning_rate=lr_schedule)

### Generator

In [ ]:
mdl_generator = models.Sequential()

mdl_generator.add(layers.Dense(n_nodes, input_dim = noise_size))
mdl_generator.add(layers.LeakyReLU(negative_slope=0.2))
mdl_generator.add(layers.Reshape((7, 7, 128)))
mdl_generator.add(layers.Conv2DTranspose(128, (4, 4), strides=(2,2), padding="same"))
mdl_generator.add(layers.LeakyReLU(negative_slope=0.2))
mdl_generator.add(layers.Conv2DTranspose(128, (4, 4), strides=(2,2), padding="same"))
mdl_generator.add(layers.LeakyReLU(negative_slope=0.2))
mdl_generator.add(layers.Conv2D(1, (7, 7), activation='sigmoid', padding="same"))

#mdl_generator.compile(optimizer="adam", loss="categorical_cross_entropy")
#mdl_generator.summary()

### Discriminator

In [ ]:
mdl_discriminator = models.Sequential()

mdl_discriminator.add(layers.Input(shape=(28, 28, 1)))
mdl_discriminator.add(layers.Conv2D(64, (3, 3), strides=(2,2), padding="same"))
mdl_discriminator.add(layers.LeakyReLU(negative_slope=0.2))
mdl_discriminator.add(layers.Dropout(0.4))
mdl_discriminator.add(layers.Conv2D(64, (3, 3), strides=(2,2), padding="same"))
mdl_discriminator.add(layers.LeakyReLU(negative_slope=0.2))
mdl_discriminator.add(layers.Dropout(0.4))
mdl_discriminator.add(layers.Flatten())
mdl_discriminator.add(layers.Dense(1, activation='sigmoid'))

mdl_discriminator.compile(optimizer=opt_adam, loss="binary_crossentropy", metrics=["accuracy"])
#mdl_discriminator.summary()

### Gan

In [ ]:
mdl_discriminator.trainable = False

mdl_gan = models.Sequential([
    mdl_generator,
    mdl_discriminator
])
opt = optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
mdl_gan.compile(loss='binary_crossentropy', optimizer=opt)

# mdl_gan = models.Sequential()
# mdl_gan.add(mdl_generator)
# mdl_gan.add(mdl_discriminator)
# mdl_gan.compile(optimizer=opt_adam, loss="binary_crossentropy")
# mdl_gan.summary()

# Dataset

### Real

In [ ]:
(x_train, _), (_, _) = mnist.load_data()
real_data = np.reshape(x_train, (len(x_train), 28, 28, 1))
real_data = real_data.astype('float32')
real_data = real_data / 255

In [ ]:
print(len(real_data))
print(real_data[0].shape)

In [ ]:
def real_sample(count) :
    i = np.random.randint(0, real_data.shape[0], count)
    x = real_data[i]
    y = np.ones((count, 1))
    return x,y

In [ ]:
x,y = real_sample(2)
print(x.shape)
print(y.shape)

### Fake

In [ ]:
def fake_noise(count) :
    i = np.random.rand(noise_size * count)
    o = i.reshape(count, noise_size)
    return o

In [ ]:
o = fake_noise(2)
print(o.shape)

In [ ]:
def fake_sample(count) :
    n = fake_noise(count)
    x = mdl_generator.predict(n)
    y = np.zeros((count, 1))
    return x,y

In [ ]:
x,y = fake_sample(2)
print(x.shape)
print(y.shape)

# Image

In [ ]:
def save_plot(examples, epoch, n=10):
	for i in range(n * n):
		plt.subplot(n, n, 1 + i)
		plt.axis('off')
		plt.imshow(examples[i, :, :, 0], cmap='gray_r')
	filename = 'generated_plot_e%03d.png' % (epoch+1)
	plt.savefig(filename)
	plt.close()

In [ ]:
def summarize_performance(epoch, n_samples=100):
    X_real, y_real = real_sample(n_samples)
    _, acc_real = mdl_discriminator.evaluate(X_real, y_real, verbose=0)
    x_fake, y_fake = fake_sample(n_samples)
    _, acc_fake = mdl_discriminator.evaluate(x_fake, y_fake, verbose=0)
    print(f'>Accuracy real: {acc_real*100}, fake: {acc_fake*100}')
    save_plot(x_fake, epoch)
    filename = f'generator_model_{epoch + 1}.h5'
    mdl_gan.save(filename)

# Train

In [ ]:
bat_per_epo = int(real_data.shape[0] / BATCH_SIZE)
half_batch = int(BATCH_SIZE/2)

for i in range(EPOCHS):
    for j in range(bat_per_epo):
        #---train mdl_discriminator
        x_real, y_real = real_sample(half_batch)
        x_fake, y_fake = fake_sample(half_batch)
        x, y = np.vstack((x_real, x_fake)), np.vstack((y_real, y_fake))
        mdl_discriminator_loss, _ = mdl_discriminator.train_on_batch(x, y)
        #---train mdl_generator
        x_gan = fake_noise(BATCH_SIZE)
        y_gan = np.ones((BATCH_SIZE, 1))
        mdl_gan_loss = mdl_gan.train_on_batch(x_gan, y_gan)
        #---display
        print(f">{i+1}, {j+1}/{bat_per_epo}, discriminator_loss={mdl_discriminator_loss:.3f}, generator_loss={mdl_gan_loss:.3f}")
    if (i+1) % 10 == 0:
        summarize_performance(i)

# Evaluation

### Loss and Accuracy

### History